In [65]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv("UrbanCabFare.csv")

features = [
    "Hour_of_Day",
    "Distance_km",
    "Demand_Supply_Ratio",
    "Trip_Duration",
    "Surge_Multiplier"
]

X = df[features]
y = df["Final_Fare"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,random_state=1832
)

In [66]:
models = {
    "OLS": LinearRegression(),

    "Ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=0.0001))
    ]),

    "Lasso": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Lasso(alpha=0.911163, max_iter=20000))
    ])
}

for model in models.values():
    model.fit(X_train, y_train)

In [67]:
results = []

n = len(y_test)
p = X_test.shape[1]   # 7 for all models

for name, model in models.items():

    pred = model.predict(X_test)

    r2 = r2_score(y_test, pred)
    adjusted_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

    results.append({
        "Model": name,
        "R2": r2,
        "Adjusted_R2": adjusted_r2,
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "MAE": mean_absolute_error(y_test, pred)
    })

results_df = (
    pd.DataFrame(results)
    .sort_values("Adjusted_R2", ascending=False)
    .reset_index(drop=True)
)

results_df

,Model,R2,Adjusted_R2,RMSE,MAE
0,Ridge,0.918044,0.916651,53.807131,42.956981
1,OLS,0.918044,0.916651,53.807137,42.956989
2,Lasso,0.918019,0.916625,53.815567,42.859484
